In [ ]:
import wrds

conn=wrds.Connection(wrds_username='cbruce1')

# Get accounting ratios
full_gvkeys = [str.zfill(gvkey, 6) for gvkey in global_universe['gvkey'].astype(float).astype(int).astype(str).unique()]
table = 'funda'
varlist = ['gvkey', 'datadate', 'at', 'sale', 'ebitda', 'ebit']
start_date = f'{start_year}-01-01'
end_date = f'{end_year}-12-30'

download_wrds_data = True
# Download from WRDS
if download_wrds_data:
    na_call = "SELECT " + ', '.join(varlist + ['ni']) + '\n' + "FROM comp_na_daily_all." + table + '\n'+ """
        WHERE gvkey IN %(gvkey_list)s
        AND datadate BETWEEN %(start_date)s AND %(end_date)s
    """
    compustat_na = conn.raw_sql(
        na_call,
        params={
                "gvkey_list": tuple(full_gvkeys),
                "start_date": start_date,
                "end_date": end_date}
    )
    df = pd.DataFrame(compustat_na)
    
    global_call = "SELECT " + ', '.join(varlist + ['nicon']) + '\n' + "FROM comp_global_daily.g_" + table + '\n' + """     
        WHERE gvkey IN %(gvkey_list)s
        AND datadate BETWEEN %(start_date)s AND %(end_date)s
    """
    compustat_global = conn.raw_sql(
        global_call,
        params={
                "gvkey_list": tuple(full_gvkeys),
                "start_date": start_date,
                "end_date": end_date}
    )
    df2 = pd.DataFrame(compustat_global)
    
    # rename col in df2 called nicon to ni
    df2 = df2.rename(columns={'nicon': 'ni'})

    # Concatenate output
    dt = pd.concat([df, df2])
    dt = dt[(dt['at']>0) & (dt['sale']>0)]

    dt['roa0'] = dt['ebitda']/dt['at']
    dt['roa1'] = dt['ebit']/dt['at']
    dt['roa2'] = dt['ni']/dt['at']
    dt['ros0'] = dt['ebitda']/dt['sale']
    dt['ros1'] = dt['ebit']/dt['sale']
    dt['ros2'] = dt['ni']/dt['sale']
    
    dt['sales_intensity'] = dt['sale']/dt['at']
    
    dt['year'] = pd.to_datetime(dt['datadate']).dt.year
    dt = dt.drop(columns=['ni', 'ebitda', 'at', 'sale', 'datadate'])

    # Save to disk
    print('Saving to disk!')
    dt.to_csv('./data/acc_comp.csv', index=False)
    
else:
    print("Read CSV")
    dt = pd.read_csv('./data/acc_comp.csv')

#Remove the ".0" In GVKEY
dt['gvkey'] = dt['gvkey'].astype(str).str.replace(r'\.0$', '', regex=True)
import numpy as np
dt["year"] = dt["year"].astype(np.int64)

#convert gvkey removing any ".0" and adding 00 to the from ensuring 6 numbers
global_universe['gvkey'] = global_universe['gvkey'].astype(str).str.replace(r'\.0$', '', regex=True).str.zfill(6)

# Merge data with global universe
global_universe = pd.merge(
    global_universe, 
    dt, 
    left_on=['gvkey', 'last_year'], 
    right_on=['gvkey', 'year'], 
    how='left'
)


Number of gvkeys with certain lens (WRSS only understands 6 digits)

In [ ]:
dt["gvkey"].astype(str).str.len().value_counts()   

With and without adding 0s to make the gvkeys 6 digits, 890 to 000890

In [ ]:
global_universe["gvkey"].astype(str).str.len().value_counts()

# how many unique gvkeys overlap *after* standardizing?
gu = global_universe["gvkey"].astype(str).str.replace(r"\.0$", "", regex=True).str.zfill(6)
dd = dt["gvkey"].astype(str).str.replace(r"\.0$", "", regex=True).str.zfill(6)
len(set(gu) & set(dd)), len(set(gu)), len(set(dd))

Missing GVkeys Before and after adding the 6 fill

They should be the same as we zfill(6) global universe at the top anyway.

In [ ]:
gu6 = global_universe["gvkey"].astype(str).str.replace(r"\.0$", "", regex=True).str.zfill(6)
gpppp = global_universe["gvkey"].astype(str).str.replace(r"\.0$", "", regex=True)
dt6 = dt["gvkey"].astype(str).str.replace(r"\.0$", "", regex=True).str.zfill(6)
missing_gvkeys_after_z_fill = set(gu6) - set(dt6)
missing_gvkeys = set(gpppp) - set(dt6)


len(missing_gvkeys_after_z_fill), len(missing_gvkeys)

Count how many non na counts after the merge

In [ ]:
match_rate2 = global_universe.assign(gvkey6=gu6).merge(
    dt.assign(gvkey6=dt6)[["gvkey6","year"]].drop_duplicates(),
    left_on=["gvkey6","last_year"],
    right_on=["gvkey6","year"],
    how="left",
    indicator=True
)["_merge"].value_counts(normalize=True)*100
match_rate2

Check dtypes in dt and global for the year so that they are the same

In [ ]:
type(global_universe["last_year"].iloc[0]), type(dt["year"].iloc[0])

Common GVKEYS in both Datasets

In [ ]:
# Amount global Universe vs the intersection of GVKEYS in both Datasets
len(set(global_universe['gvkey'])), len(set(global_universe['gvkey']).intersection(set(dt['gvkey'])))

Number of Nas in all accounting columns

In [ ]:
cols = ["roa0","roa1","roa2","ros0","ros1","ros2","sales_intensity"]
(100 * global_universe[cols].isna().mean()).sort_values(ascending=False)

In [ ]:
#global_universe.to_csv("./debuggggg.csv")